Last edited: 1/28/2026, 3:38 PM CST
# Benchmarking MLE algorithm

In [1]:
# import base packages for data analysis
import pandas as pd
import xarray as xr

from config import DATA_ROOT
from src.utils import extract_model_name
from src.cmip_dataclass import CMIP6EnsembleConfig

## Global attributes

In [2]:
fit = 'nonstat'
anom_types = ['raw', 'trend', 'annmean']
TMIN = 1979

## ERA5

In [3]:
era5_variable = 't2m_annual_max'
era5_grid = '1deg'

era5_perf_stats = {}
era5_perf_stats['MLE_success_rate'] = {}

In [13]:
for anom_type in anom_types:
    tmp_ds = xr.open_dataset(
        DATA_ROOT / 'ERA5'/ 'gev' / f'era5_{era5_variable}_{era5_grid}_landonly_gev_{fit}_TMIN{TMIN}_{anom_type}.nc'
    )

    era5_perf_stats['MLE_success_rate'][anom_type] = tmp_ds.MLE_success_rate
    
    tmp_ds.close()

In [26]:
era5_df = pd.DataFrame(era5_perf_stats)
era5_df.to_csv(
    DATA_ROOT / 'stats' / 'mle_perf_era5.csv'
)

## CMIP6

In [32]:
cmip_variable = 'tas_annual_max'

CMIPConfig = CMIP6EnsembleConfig.from_yaml(
    'config/meta.yaml',
    'config/qc.yaml'
)

cmip_perf_stats = {}
cmip_perf_stats['MLE_success_rate'] = {}

In [33]:
for anom_type in anom_types:
    # tmp empty list
    tmp_success_rates = []

    # make file/model matcher for 
    data_path = DATA_ROOT / 'CMIP6' / cmip_variable / 'gev'

    # Make all landonly file names
    fnames = [f for f in data_path.glob(f"*{fit}*{anom_type}*.nc") if "allmems" not in f.name]  # screen out allmems results

    modelname_filepath_matcher = {
    extract_model_name(f): f for f in fnames
    }

    for m in CMIPConfig.iter_active_models(cmip_variable):
        fname = modelname_filepath_matcher[m.name]
        tmp_ds = xr.open_dataset(fname)

        tmp_success_rates.append(tmp_ds.MLE_success_rate)

        tmp_ds.close()

    cmip_perf_stats['MLE_success_rate'][anom_type] = tmp_success_rates

In [37]:
cmip_perf_stats['MLE_success_rate']

{'raw': [0.9881462569526762,
  0.8044132397191575,
  0.994301085073402,
  0.8620406674569162,
  0.996626242363454,
  0.9948481809063554,
  0.9933436673657335,
  0.9823561593872526,
  0.9912464666727455,
  0.8136645962732919,
  0.9935716239627975,
  0.9940731284763381,
  0.9916567885474605,
  0.9912464666727455,
  0.9920215191027628,
  0.9919303364639372,
  0.993890763198687,
  0.9849092732743686,
  0.979757454180724,
  0.9910185100756815,
  0.9942554937539893,
  0.9834959423725722,
  0.993480441323972,
  0.9812619677213459,
  0.9876447524391356,
  0.9916567885474605,
  0.9898331357709492,
  0.9914288319503967,
  0.9736026260599981],
 'trend': [0.9886477614662168,
  0.8071487188839245,
  0.9939819458375125,
  0.8616303455822012,
  0.9967630163216924,
  0.9939363545180998,
  0.9941187197957508,
  0.9828120725813805,
  0.9920215191027628,
  0.8156113840734217,
  0.993252484726908,
  0.9942554937539893,
  0.991565605908635,
  0.9909273274368561,
  0.9930245281298441,
  0.991793562505699,
 

In [45]:
cmip_df = pd.DataFrame.from_dict(cmip_perf_stats['MLE_success_rate'], orient='index',
                                 columns=[m.name for m in CMIPConfig.iter_active_models(cmip_variable)])
cmip_df

,AWI-CM-1-1-MR,BCC-CSM2-MR,CAMS-CSM1-0,CESM2-WACCM,CMCC-CM2-SR5,CMCC-ESM2,CNRM-CM6-1-HR,CNRM-ESM2-1,EC-Earth3-CC,EC-Earth3-Veg,...,KACE-1-0-G,KIOST-ESM,MIROC-ES2L,MIROC6,MPI-ESM1-2-HR,MPI-ESM1-2-LR,NorCPM1,NorESM2-LM,TaiESM1,UKESM1-0-LL
raw,0.988146,0.804413,0.994301,0.862041,0.996626,0.994848,0.993344,0.982356,0.991246,0.813665,...,0.991019,0.994255,0.983496,0.993480,0.981262,0.987645,0.991657,0.989833,0.991429,0.973603
trend,0.988648,0.807149,0.993982,0.861630,0.996763,0.993936,0.994119,0.982812,0.992022,0.815611,...,0.992022,0.994620,0.982630,0.993161,0.981216,0.984089,0.991566,0.990289,0.991019,0.975563
annmean,0.993025,0.800629,0.994939,0.842983,0.993936,0.994119,0.994985,0.982219,0.993389,0.817512,...,0.993800,0.994483,0.983359,0.994028,0.983587,0.993298,0.994529,0.993936,0.991839,0.979347


In [47]:
cmip_df.mean(axis=1)

raw        0.972818
trend      0.972978
annmean    0.974040
dtype: float64

In [48]:
cmip_df.to_csv(
    DATA_ROOT / 'stats' / 'mle_perf_cmip.csv'
)